# Conflictos en Git y GitHub
**Ciencia de Datos · Programación II · Universidad Externado de Colombia**

Este cuaderno continúa la **Subsección 4 del Taller preparcial (Ramas y viaje por las versiones)**. Allí usted trabajó con dos ramas, `main` y `analisis-extra`, y las unió con `git merge` **sin problemas**, porque cada rama modificó partes distintas del cuaderno. Aquí va a provocar a propósito lo que ocurre cuando **las dos ramas modifican las mismas líneas**: un **conflicto**, y aprenderá a resolverlo.

## Instrucciones
1. Suba este cuaderno a **su repositorio del taller** (el que ya tiene las ramas `main` y `analisis-extra`) y ábralo en **GitHub Codespaces**.
2. Los comandos que **cambian de rama o unen ramas** (`git switch`, `git merge`) ejecútelos **siempre desde la terminal** (`Ctrl + ñ` o *Terminal → New Terminal*).
3. Los comandos que **solo consultan** (`git status`, `git log`, `git diff`) y `python analisis_salarios.py` puede ejecutarlos desde las celdas, anteponiendo `!`.
4. En esta práctica **no se hace commit de este cuaderno**, solo del archivo `analisis_salarios.py`. Por eso use siempre `git add analisis_salarios.py`, **nunca** `git add .`.

> **¿Por qué un archivo `.py` y no un cuaderno?** Un `.ipynb` es por dentro un archivo JSON con código, salidas y metadatos. Un conflicto en un cuaderno es muy difícil de leer. En un `.py` cada línea es código, así que el conflicto se entiende de inmediato.

---
## Parte 1: ¿Qué es un conflicto?

Git guarda la historia como una serie de commits. Cuando se unen dos historias que avanzaron por separado (dos ramas, o su copia local y la de GitHub), Git intenta combinar los cambios automáticamente:

- **Cambios en archivos distintos, o en líneas distintas del mismo archivo** → Git los combina solo. No hay conflicto (esto fue lo que pasó en el punto 8 del taller).
- **Cambios en las mismas líneas, de forma distinta** → Git **no puede adivinar** cuál versión es la correcta. Se detiene y le pide a usted que decida. Eso es un **conflicto**.

```
                 C3   ← main:            PORCENTAJE_BONO = 0.12
                /
C0 ── C1 ── C2 ─
                \
                 C4   ← analisis-extra:  PORCENTAJE_BONO = 0.15
```

Al ejecutar `git merge analisis-extra` estando en `main`, Git no sabe si dejar `0.12` o `0.15`.

**¿Cuándo aparece un conflicto?**

| Situación | Qué pasa |
|---|---|
| `git merge <rama>` | Las dos ramas cambiaron las mismas líneas. |
| `git pull` | Alguien subió a GitHub cambios en las mismas líneas que usted cambió en su copia. |
| **Pull Request** en GitHub | GitHub avisa *"This branch has conflicts that must be resolved"*. |

### ¿Cómo se ve un conflicto?

Git deja **las dos versiones** dentro del archivo, separadas por marcadores:

```
<<<<<<< HEAD
PORCENTAJE_BONO = 0.12
=======
PORCENTAJE_BONO = 0.15
>>>>>>> analisis-extra
```

| Marcador | Significado |
|---|---|
| `<<<<<<< HEAD` | Empieza **su versión**, la de la rama en la que está parado (*Current Change*). |
| `=======` | Separa las dos versiones. |
| `>>>>>>> analisis-extra` | Termina **la versión que llega** de la otra rama (*Incoming Change*). |

**Resolver** un conflicto es: dejar el contenido correcto, **borrar los marcadores**, y hacer `git add` + `git commit`. Mientras el conflicto no esté resuelto, el archivo no funciona (los marcadores no son código Python válido).

### El visor de conflictos de VS Code / Codespaces

Cuando hay un conflicto, Codespaces lo muestra así:

1. En el panel **Source Control** (`Ctrl + Shift + G`) aparece la sección **Merge Changes** con los archivos en conflicto (marcados con **!**).
2. Al abrir el archivo, encima de cada conflicto aparecen los botones:
   ```
   Accept Current Change | Accept Incoming Change | Accept Both Changes | Compare Changes
   ```
3. Abajo a la derecha aparece el botón **Resolve in Merge Editor**, que abre un visor de tres paneles:
   ```
   ┌──────────────────────────┬──────────────────────────┐
   │ Incoming (analisis-extra)│ Current (main)           │
   ├──────────────────────────┴──────────────────────────┤
   │ Result: lo que quedará en el archivo                │
   └─────────────────────────────────────────────────────┘
   ```
   Se marcan las casillas de lo que se quiere conservar, se ajusta el panel **Result** si hace falta y se termina con **Complete Merge**.

> Si se arrepiente a mitad de camino, `git merge --abort` deja todo como estaba antes del merge.

---
## Parte 2: Preparar el archivo en las dos ramas

**1.** Desde la terminal, verifique que está en `main` y que está al día con GitHub:

```bash
git switch main
git pull
git status
```

Ejecute en la celda siguiente `!git branch` y confirme que existen las ramas `main` y `analisis-extra`.

> Si no hizo la Subsección 4 del taller y no tiene la rama `analisis-extra`, créela con `git branch analisis-extra` y súbala con `git push -u origin analisis-extra`.

In [1]:
# Escriba aquí su comando (con ! al inicio para ejecutarlo desde el cuaderno)
!git branch

  analisis-extra
* main


**2.** Ejecute la siguiente celda. El comando mágico `%%writefile` **crea el archivo** `analisis_salarios.py` con el contenido de la celda. El script usa los datos ya limpios de la Subsección 1 del taller, calcula un **bono** y marca a quienes tienen **salario alto**, según dos parámetros:

- `PORCENTAJE_BONO = 0.10`
- `UMBRAL_SALARIO_ALTO = 5.0`

In [2]:
%%writefile analisis_salarios.py
"""Análisis de salarios de los empleados (datos ya limpios de la Subsección 1 del taller)."""
import pandas as pd

# Parámetros del análisis
PORCENTAJE_BONO = 0.10
UMBRAL_SALARIO_ALTO = 5.0

datos = {
    "Nombre": ["Mariana", "Esteban", "Camilo", "Juliana", "Nicolas", "Paula",
               "Sebastian", "Laura", "Andres", "Catalina", "Manuela"],
    "Genero": ["Mujer", "Hombre", "Hombre", "Mujer", "Hombre", "Mujer",
               "Hombre", "Mujer", "Hombre", "Mujer", "Mujer"],
    "Departamento": ["Marketing", "Ventas", "IT", "Marketing", "Ventas", "Marketing",
                     "IT", "Marketing", "Finanzas", "Marketing", "Finanzas"],
    "Salario": [3.8, 4.2, 6.0, 3.2, 4.0, 3.9, 6.3, 3.7, 4.5, 5.5, 3.6],
    "AñosExperiencia": [2, 5, 10, 3, 7, 4, 15, 3, 9, 6, 8],
}
df = pd.DataFrame(datos)

# Nuevas variables
df["Bono"] = df["Salario"] * PORCENTAJE_BONO
df["SalarioAlto"] = df["Salario"] > UMBRAL_SALARIO_ALTO

# Resumen por departamento
resumen = df.groupby("Departamento")["Salario"].mean()

print(df[["Nombre", "Departamento", "Salario", "Bono", "SalarioAlto"]])
print("\nResumen por departamento:")
print(resumen)


Writing analisis_salarios.py


Ejecute el script para verificar que funciona:

In [3]:
!python analisis_salarios.py

       Nombre Departamento  Salario  Bono  SalarioAlto
0     Mariana    Marketing      3.8  0.38        False
1     Esteban       Ventas      4.2  0.42        False
2      Camilo           IT      6.0  0.60         True
3     Juliana    Marketing      3.2  0.32        False
4     Nicolas       Ventas      4.0  0.40        False
5       Paula    Marketing      3.9  0.39        False
6   Sebastian           IT      6.3  0.63         True
7       Laura    Marketing      3.7  0.37        False
8      Andres     Finanzas      4.5  0.45        False
9    Catalina    Marketing      5.5  0.55         True
10    Manuela     Finanzas      3.6  0.36        False

Resumen por departamento:
Departamento
Finanzas     4.05
IT           6.15
Marketing    4.02
Ventas       4.10
Name: Salario, dtype: float64


**3.** Desde la terminal, haga commit del archivo en `main` y súbalo:

```bash
git add analisis_salarios.py
git commit -m "Conflicto - Punto 2: crear analisis_salarios.py"
git push
```

**4.** Ahora lleve ese archivo a la rama `analisis-extra`. Cámbiese a ella y **traiga los cambios de `main`**:

```bash
git switch analisis-extra
git merge main
git push
```

Como `analisis-extra` no tiene commits nuevos, Git solo **adelanta** la rama hasta `main` (un *fast-forward*), sin conflicto. Ejecute `!git log --oneline --graph --all -5` y responda: ¿en qué commit están ahora `main` y `analisis-extra`?

In [8]:
# Escriba aquí su comando (con ! al inicio para ejecutarlo desde el cuaderno)
!git log --oneline --graph --all --decorate -5

* 0994fa0 (HEAD -> analisis-extra, origin/main, origin/analisis-extra, origin/HEAD, main) Conflicto - Punto 2: crear analisis_salarios.py
* 5a9559b (tag: v1.0-preparcial) Subseccion 4 - Punto 13: etiqueta de la version final
* 2042f13 Subseccion 4 - Punto 12: deshacer un error con git revert
* 7369f22 Revert "Error: borré una solución por accidente"
* b043504 Error: borré una solución por accidente


**Respuesta:** _escriba aquí su respuesta_

---
## Parte 3: Cambiar las mismas líneas en las dos ramas

**5.** Verifique en la terminal que está en `analisis-extra` (`git branch`). En esta rama el equipo decidió:

- Subir el bono al **15 %** (`PORCENTAJE_BONO = 0.15`).
- Bajar el umbral de salario alto a **4.5** (`UMBRAL_SALARIO_ALTO = 4.5`).
- Usar la **mediana** en el resumen por departamento.
- Agregar al final un conteo de empleados con salario alto.

Ejecute la celda para reescribir el archivo con esos cambios y compruebe que funciona:

In [9]:
%%writefile analisis_salarios.py
"""Análisis de salarios de los empleados (datos ya limpios de la Subsección 1 del taller)."""
import pandas as pd

# Parámetros del análisis
PORCENTAJE_BONO = 0.15
UMBRAL_SALARIO_ALTO = 4.5

datos = {
    "Nombre": ["Mariana", "Esteban", "Camilo", "Juliana", "Nicolas", "Paula",
               "Sebastian", "Laura", "Andres", "Catalina", "Manuela"],
    "Genero": ["Mujer", "Hombre", "Hombre", "Mujer", "Hombre", "Mujer",
               "Hombre", "Mujer", "Hombre", "Mujer", "Mujer"],
    "Departamento": ["Marketing", "Ventas", "IT", "Marketing", "Ventas", "Marketing",
                     "IT", "Marketing", "Finanzas", "Marketing", "Finanzas"],
    "Salario": [3.8, 4.2, 6.0, 3.2, 4.0, 3.9, 6.3, 3.7, 4.5, 5.5, 3.6],
    "AñosExperiencia": [2, 5, 10, 3, 7, 4, 15, 3, 9, 6, 8],
}
df = pd.DataFrame(datos)

# Nuevas variables
df["Bono"] = df["Salario"] * PORCENTAJE_BONO
df["SalarioAlto"] = df["Salario"] > UMBRAL_SALARIO_ALTO

# Resumen por departamento
resumen = df.groupby("Departamento")["Salario"].median()

print(df[["Nombre", "Departamento", "Salario", "Bono", "SalarioAlto"]])
print("\nResumen por departamento:")
print(resumen)
print("\nEmpleados con salario alto:", df["SalarioAlto"].sum())


Overwriting analisis_salarios.py


In [10]:
!python analisis_salarios.py

       Nombre Departamento  Salario   Bono  SalarioAlto
0     Mariana    Marketing      3.8  0.570        False
1     Esteban       Ventas      4.2  0.630        False
2      Camilo           IT      6.0  0.900         True
3     Juliana    Marketing      3.2  0.480        False
4     Nicolas       Ventas      4.0  0.600        False
5       Paula    Marketing      3.9  0.585        False
6   Sebastian           IT      6.3  0.945         True
7       Laura    Marketing      3.7  0.555        False
8      Andres     Finanzas      4.5  0.675        False
9    Catalina    Marketing      5.5  0.825         True
10    Manuela     Finanzas      3.6  0.540        False

Resumen por departamento:
Departamento
Finanzas     4.05
IT           6.15
Marketing    3.80
Ventas       4.10
Name: Salario, dtype: float64

Empleados con salario alto: 3


Haga commit en `analisis-extra` y súbalo:

```bash
git add analisis_salarios.py
git commit -m "Conflicto - Punto 5: bono 15% y mediana en analisis-extra"
git push
```

**6.** Regrese a `main` **desde la terminal**:

```bash
git switch main
```

Observe que `analisis_salarios.py` volvió a la versión original (bono 10 %). En `main`, **otra persona** decidió algo distinto:

- Bono del **12 %** (`PORCENTAJE_BONO = 0.12`).
- Umbral de salario alto de **5.5** (`UMBRAL_SALARIO_ALTO = 5.5`).
- Mostrar en el resumen el **promedio y el máximo**.

Ejecute la celda, compruebe que funciona y haga commit en `main`:

In [11]:
%%writefile analisis_salarios.py
"""Análisis de salarios de los empleados (datos ya limpios de la Subsección 1 del taller)."""
import pandas as pd

# Parámetros del análisis
PORCENTAJE_BONO = 0.12
UMBRAL_SALARIO_ALTO = 5.5

datos = {
    "Nombre": ["Mariana", "Esteban", "Camilo", "Juliana", "Nicolas", "Paula",
               "Sebastian", "Laura", "Andres", "Catalina", "Manuela"],
    "Genero": ["Mujer", "Hombre", "Hombre", "Mujer", "Hombre", "Mujer",
               "Hombre", "Mujer", "Hombre", "Mujer", "Mujer"],
    "Departamento": ["Marketing", "Ventas", "IT", "Marketing", "Ventas", "Marketing",
                     "IT", "Marketing", "Finanzas", "Marketing", "Finanzas"],
    "Salario": [3.8, 4.2, 6.0, 3.2, 4.0, 3.9, 6.3, 3.7, 4.5, 5.5, 3.6],
    "AñosExperiencia": [2, 5, 10, 3, 7, 4, 15, 3, 9, 6, 8],
}
df = pd.DataFrame(datos)

# Nuevas variables
df["Bono"] = df["Salario"] * PORCENTAJE_BONO
df["SalarioAlto"] = df["Salario"] > UMBRAL_SALARIO_ALTO

# Resumen por departamento
resumen = df.groupby("Departamento")["Salario"].agg(["mean", "max"])

print(df[["Nombre", "Departamento", "Salario", "Bono", "SalarioAlto"]])
print("\nResumen por departamento:")
print(resumen)


Overwriting analisis_salarios.py


In [12]:
!python analisis_salarios.py

       Nombre Departamento  Salario   Bono  SalarioAlto
0     Mariana    Marketing      3.8  0.456        False
1     Esteban       Ventas      4.2  0.504        False
2      Camilo           IT      6.0  0.720         True
3     Juliana    Marketing      3.2  0.384        False
4     Nicolas       Ventas      4.0  0.480        False
5       Paula    Marketing      3.9  0.468        False
6   Sebastian           IT      6.3  0.756         True
7       Laura    Marketing      3.7  0.444        False
8      Andres     Finanzas      4.5  0.540        False
9    Catalina    Marketing      5.5  0.660        False
10    Manuela     Finanzas      3.6  0.432        False

Resumen por departamento:
              mean  max
Departamento           
Finanzas      4.05  4.5
IT            6.15  6.3
Marketing     4.02  5.5
Ventas        4.10  4.2


```bash
git add analisis_salarios.py
git commit -m "Conflicto - Punto 6: bono 12% y promedio y maximo en main"
git push
```

Ejecute `!git log --oneline --graph --all -6`. ¿Cuántos commits tiene cada rama que la otra no tiene?

In [13]:
# Escriba aquí su comando (con ! al inicio para ejecutarlo desde el cuaderno)
!git log --oneline --graph --all --decorate -6

* 57f458f (HEAD -> main, origin/main, origin/HEAD) Conflicto - Punto 6: bono 12% y promedio y maximo en main
| * 5be620a (analisis-extra) Conflicto - Punto 5: bono 15% y mediana en analisis-extra
|/  
* 0994fa0 (origin/analisis-extra) Conflicto - Punto 2: crear analisis_salarios.py
* 5a9559b (tag: v1.0-preparcial) Subseccion 4 - Punto 13: etiqueta de la version final
* 2042f13 Subseccion 4 - Punto 12: deshacer un error con git revert
* 7369f22 Revert "Error: borré una solución por accidente"


**Respuesta:** _escriba aquí su respuesta_

---
## Parte 4: Provocar y resolver el conflicto

**7.** Estando en `main`, intente unir la rama `analisis-extra` **desde la terminal**:

```bash
git merge analisis-extra
```

Git responderá algo como:

```
Auto-merging analisis_salarios.py
CONFLICT (content): Merge conflict in analisis_salarios.py
Automatic merge failed; fix conflicts and then commit the result.
```

Ejecute `!git status` en la celda siguiente y explique qué dice sobre `analisis_salarios.py`.

In [14]:
# Escriba aquí su comando (con ! al inicio para ejecutarlo desde el cuaderno)
!git status


On branch main
Your branch is up to date with 'origin/main'.

You have unmerged paths.
  (fix conflicts and run "git commit")
  (use "git merge --abort" to abort the merge)

Unmerged paths:
  (use "git add <file>..." to mark resolution)
	both modified:   analisis_salarios.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	conflicto.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


**Respuesta:** _escriba aquí su respuesta_

**8.** Ejecute el script **sin resolver** el conflicto. ¿Qué error aparece y por qué?

In [ ]:
!python analisis_salarios.py

**Respuesta:** _escriba aquí su respuesta_

**9.** Abra `analisis_salarios.py` en el editor (panel de archivos de la izquierda). Verá los marcadores `<<<<<<<`, `=======` y `>>>>>>>` resaltados en colores y los botones *Accept Current Change | Accept Incoming Change | Accept Both Changes*.

Responda:
- ¿Cuántos bloques en conflicto hay y en qué líneas?
- La rama `analisis-extra` también **agregó una línea al final** (el conteo de empleados con salario alto). ¿Esa línea quedó en conflicto? ¿Por qué Git sí pudo combinarla solo?

**Respuesta:** _escriba aquí su respuesta_

**10.** Resuelva el conflicto con el **Merge Editor**: haga clic en **Resolve in Merge Editor** (abajo a la derecha del archivo). La decisión del equipo es:

- **Bono:** el de `main` (**0.12**).
- **Umbral de salario alto:** el de `analisis-extra` (**4.5**).
- **Resumen por departamento:** promedio, mediana y máximo, `agg(["mean", "median", "max"])`.

Como el primer bloque mezcla una línea de cada rama y el segundo combina las dos ideas, **ningún botón por sí solo sirve**. Marque las casillas que le sirvan y **edite a mano el panel Result** hasta que quede así:

```python
PORCENTAJE_BONO = 0.12
UMBRAL_SALARIO_ALTO = 4.5
...
resumen = df.groupby("Departamento")["Salario"].agg(["mean", "median", "max"])
```

Termine con **Complete Merge**. Verifique que el archivo **no tiene marcadores** y que el script funciona:

In [ ]:
!python analisis_salarios.py

> **Cuidado con *Accept Both Changes*:** en el bloque de los parámetros dejaría **dos veces** `PORCENTAJE_BONO` y `UMBRAL_SALARIO_ALTO`. El script correría sin error, pero Python usaría el **último** valor asignado, que tal vez no es el que usted quería. Resolver un conflicto no es solo quitar los marcadores: hay que **decidir** qué debe quedar.

**11.** Termine el merge desde la terminal:

```bash
git add analisis_salarios.py
git commit -m "Conflicto - Punto 11: resolver conflicto entre main y analisis-extra"
git push
```

Ejecute `!git log --oneline --graph --all -8` y describa cómo se ve en el grafo el **commit de merge**.

In [ ]:
# Escriba aquí su comando (con ! al inicio para ejecutarlo desde el cuaderno)


**Respuesta:** _escriba aquí su respuesta_

---
## Parte 5 (reto): resolver un conflicto desde GitHub con un Pull Request

En equipos de trabajo las ramas se unen desde GitHub con un **Pull Request (PR)**. GitHub también detecta los conflictos y permite resolver los sencillos desde el navegador.

**12.** Actualice `analisis-extra` con `main` y cree un nuevo conflicto:

```bash
git switch analisis-extra
git merge main                      # fast-forward: quedan iguales
```

- En `analisis-extra` cambie a mano en el archivo `PORCENTAJE_BONO = 0.20`, haga commit y `git push`.
- Pase a `main` (`git switch main`), cambie `PORCENTAJE_BONO = 0.08`, haga commit y `git push`.

**13.** En GitHub, vaya a la pestaña **Pull requests → New pull request**, con **base: `main`** y **compare: `analisis-extra`**, y cree el PR. GitHub mostrará: *"This branch has conflicts that must be resolved"*.

**14.** Haga clic en **Resolve conflicts**. Se abre un editor web con los mismos marcadores. Deje el valor que decida el equipo, **borre los marcadores**, haga clic en **Mark as resolved** y luego en **Commit merge**. Finalmente, haga clic en **Merge pull request**.

**15.** De vuelta en Codespaces, traiga el resultado a su copia local con `git pull` y verifique el grafo con `!git log --oneline --graph --all -10`. ¿Qué diferencia hay entre resolver el conflicto en Codespaces y en GitHub?

> Si el conflicto es muy complejo, GitHub no muestra el botón **Resolve conflicts** y pide resolverlo en local (o en Codespaces), como en la Parte 4.

In [ ]:
# Escriba aquí su comando (con ! al inicio para ejecutarlo desde el cuaderno)


**Respuesta:** _escriba aquí su respuesta_

---
## Resumen

| Comando / acción | Para qué sirve |
|---|---|
| `git merge <rama>` | Une la rama indicada en la actual; puede generar conflictos. |
| `git status` | Muestra los archivos en conflicto (*both modified*). |
| `<<<<<<<` / `=======` / `>>>>>>>` | Marcan las dos versiones en conflicto dentro del archivo. |
| Accept Current / Incoming / Both | Botones rápidos de VS Code para cada bloque. |
| **Resolve in Merge Editor** | Visor de tres paneles para decidir con calma. |
| `git add` + `git commit` | Marcan el conflicto como resuelto y crean el commit de merge. |
| `git merge --abort` | Cancela el merge y vuelve al estado anterior. |
| **Resolve conflicts** (GitHub) | Resolver un conflicto sencillo de un Pull Request desde el navegador. |